# 2. óra -- válogatás, szűrés, rendezés, új oszlopok

Ez az óra a `dplyr` négy alapvető igéjének (`select`, `filter`, `arrange`, `mutate`) python/pandas megfelelőit gyakorolja be a Disney-címek adattábláján.

| R / dplyr | Python / pandas |
|---|---|
| `select()` | oszloplista: `df[['a','b']]`, vagy mintaillesztés: `df.filter(regex=...)` |
| `filter()` | logikai maszk: `df[feltétel]` |
| `arrange()` | `df.sort_values()` |
| `mutate()` | `df.assign()` vagy `df['új'] = ...` |

A pandasben nincs natív pipe operátor, de a metódusláncolás (`.query().sort_values()...`) hasonló olvashatóságot ad, mint az R `|>` láncolása.

In [ ]:
# adatkezeléshez a pandas könyvtárat használjuk
import pandas as pd

# adattábla beolvasása közvetlenül a GitHub repóból, ugyanaz a forrás, mint az R órán
df = pd.read_csv("https://raw.githubusercontent.com/petyaracz/AdatmanipulacioModellezes/refs/heads/main/dat/R/disney_titles_transformed.csv")

# gyors ránézés: hány sor/oszlop, milyen típusúak az oszlopok
df.shape


In [ ]:
# első pár sor és az oszlopnevek áttekintése (df.info() a pandas megfelelője a glimpse()-nek)
df.info()


## Válogatás (`select`)

Oszlopok kiválasztásához pandasben egy oszlopnév-listát adunk meg: `df[['oszlop1', 'oszlop2']]`. Mintaillesztéshez (pl. "minden oszlop, ami `genre_`-vel kezdődik") listaértelmezést (*list comprehension*) vagy a `df.filter(regex=...)` függvényt használjuk.

In [ ]:
# cím, imdb pontszám
df[['title', 'imdb_score']]


In [ ]:
# cím, gyártó ország
df[['title', 'production_countries']]


In [ ]:
# cím, megjelenés éve
df[['title', 'release_year']]


In [ ]:
# cím, imdb dolgok (minden imdb-vel kapcsolatos oszlop)
df[['title', 'imdb_id', 'imdb_score', 'imdb_votes']]


In [ ]:
# cím, műfajok -- a műfaj-oszlopok mind 'genre_' előtaggal kezdődnek
# listaértelmezéssel gyűjtjük össze az ilyen nevű oszlopokat (ez felel meg a dplyr starts_with()-nek)
genre_cols = [c for c in df.columns if c.startswith('genre_')]
df[['title'] + genre_cols]


In [ ]:
# cím, y-ra végződő oszlopok (ez felel meg a dplyr ends_with()-nek)
y_cols = [c for c in df.columns if c.endswith('y')]
df[['title'] + y_cols]


## Szűrés (`filter`)

A `filter()` megfelelője pandasben a logikai maszkos indexelés: `df[feltétel]`, ahol `feltétel` egy igaz/hamis értékekből álló sorozat. Több feltétel kombinálásához `&` (ÉS) és `|` (VAGY) operátorokat használunk -- figyeljünk rá, hogy minden részfeltételt zárójelbe kell tenni, mert a `&`/`|` kötési erőssége más, mint a pythonos `and`/`or`-é.

In [ ]:
# csak filmek
df[df['type'] == 'MOVIE']


In [ ]:
# 1990-ben megjelent filmek
df[df['release_year'] == 1990]


In [ ]:
# 1940-ben megjelent filmek
df[df['release_year'] == 1940]


In [ ]:
# 1940 után megjelent filmek
df[df['release_year'] > 1940]


In [ ]:
# 1940 előtt megjelent filmek
df[df['release_year'] < 1940]


In [ ]:
# 1940 után, 1995 előtt megjelent filmek
df[(df['release_year'] > 1940) & (df['release_year'] < 1995)]


In [ ]:
# animációs filmek -- genre_animation egy logikai (bool) oszlop, önmagában is használható maszkként
df[df['genre_animation']]


In [ ]:
# G kategóriába sorolt és 90 percnél hosszabb filmek
df[(df['age_certification'] == 'G') & (df['runtime'] > 90)]


In [ ]:
# 1940 előtt, VAGY 1995 után megjelent filmek
df[(df['release_year'] < 1940) | (df['release_year'] > 1995)]


In [ ]:
# 1940-ben megjelent VAGY 8,0-nál nagyobb IMDb-pontszámmal rendelkező filmek
df[(df['release_year'] == 1940) | (df['imdb_score'] > 8.0)]


In [ ]:
# az 1950 előtt megjelent ÉS (fantasy VAGY 7-nél nagyobb IMDB pontszámmal rendelkező) filmek
df[(df['release_year'] < 1950) & (df['genre_fantasy'] | (df['imdb_score'] > 7))]


In [ ]:
# 1930 és 1950 között megjelent címek -- a .between() a dplyr between()-jének felel meg
df[df['release_year'].between(1930, 1950)]


In [ ]:
# családi animációk
df[df['genre_family'] & df['genre_animation']]


In [ ]:
# horror animációk
df[df['genre_horror'] & df['genre_animation']]


In [ ]:
# NA imdb -- hiányzó imdb pontszámú sorok
df[df['imdb_score'].isna()]


In [ ]:
# 120 percnél rövidebb, 1930 után megjelent, és legalább két megadott műfajhoz (animation, family, music, fantasy) tartozó címek
# a négy logikai oszlopot összeadva megkapjuk, hány műfajhoz tartozik egyszerre egy cím
mufaj_oszlopok = ['genre_animation', 'genre_family', 'genre_music', 'genre_fantasy']
hany_mufaj = df[mufaj_oszlopok].sum(axis=1)

df[(df['runtime'] < 120) & (df['release_year'] > 1930) & (hany_mufaj >= 2)]


## Rendezés (`arrange`)

A `sort_values()` az `arrange()` megfelelője. Csökkenő sorrendhez az `ascending=False` argumentumot adjuk meg (nincs külön "fordított" oszlop, mint az R `desc()`-je, de ugyanúgy használható egyben több oszlopra is).

In [ ]:
# megjelenés éve szerint növekvő sorrendben
df.sort_values('release_year')


In [ ]:
# imdb pontszám szerint növekvő sorrendben
df.sort_values('imdb_score')


In [ ]:
# imdb pontszám szerint csökkenő (fordított) sorrendben
df.sort_values('imdb_score', ascending=False)


In [ ]:
# kategória (age_certification) szerint
df.sort_values('age_certification')


## Új oszlopok létrehozása (`mutate`)

A `df.assign()` az R `mutate()`-jének felel meg: egy vagy több új oszlopot ad az adattáblához anélkül, hogy az eredetit módosítaná. (A `df['új_oszlop'] = ...` forma ezzel szemben a meglévő adattáblát módosítja a helyén -- lánc közben az `assign()` a kényelmesebb.)

In [ ]:
# perc -> óra
df.assign(runtime_hours=df['runtime'] / 60)[['title', 'runtime', 'runtime_hours']]


In [ ]:
# horror + animáció -> "horror animáció" logikai oszlop
df.assign(horror_animacio=df['genre_horror'] & df['genre_animation'])[['title', 'genre_horror', 'genre_animation', 'horror_animacio']]


In [ ]:
# "1940 után jelent meg" jelzőoszlop
df.assign(utan_1940=df['release_year'] > 1940)[['title', 'release_year', 'utan_1940']]


In [ ]:
# G kategóriába sorolt és 90 percnél hosszabb filmek jelzőoszlop
df.assign(g_es_hosszu=(df['age_certification'] == 'G') & (df['runtime'] > 90))[['title', 'age_certification', 'runtime', 'g_es_hosszu']]


In [ ]:
# 1940-ben megjelent VAGY 8,0-nál nagyobb IMDb-pontszámú filmek jelzőoszlop
df.assign(jo_vagy_regi=(df['release_year'] == 1940) | (df['imdb_score'] > 8.0))[['title', 'release_year', 'imdb_score', 'jo_vagy_regi']]


## Mutate + filter

A két igét értelemszerűen egymás után lehet alkalmazni: előbb létrehozzuk az új oszlopot `assign()`-nal, utána a rajta alapuló feltétellel szűrünk.

In [ ]:
# 1940-ben megjelent vagy 8,0-nál nagyobb IMDb-pontszámú filmek -> szűrés erre a jelzőoszlopra
df_jelzett = df.assign(jo_vagy_regi=(df['release_year'] == 1940) | (df['imdb_score'] > 8.0))
df_jelzett[df_jelzett['jo_vagy_regi']]


In [ ]:
# G kategóriába sorolt és 90 percnél hosszabb filmek -> ezek közül a családi (family) filmek
df_jelzett2 = df.assign(g_es_hosszu=(df['age_certification'] == 'G') & (df['runtime'] > 90))
df_jelzett2[df_jelzett2['g_es_hosszu'] & df_jelzett2['genre_family']]
